# CitizenTech · Monitor de Pavimento — Módulo de Visão Computacional

**Global Solution 2026.2 · FIAP** · Nicho: **Infraestrutura e Pavimentação** — detecção de buracos na pavimentação viária por caixas delimitadoras (bounding boxes).

Este notebook cobre as cinco etapas pedidas no enunciado:

| Etapa | Seção |
|---|---|
| 1. Seleção e preparação dos dados | §1 Dados |
| 2. Implementação (detector YOLO com transfer learning) | §2 Modelo e treinamento |
| 3. Avaliação (precisão, recall, mAP, matriz de confusão, análise de erros) | §3 Avaliação |
| 4. Integração ao CitizenTech (imagem + vídeo/monitoramento contínuo) | §5 Integração |
| 5. Desempenho (latência, FPS, hardware) e justificativa da arquitetura | §4 Desempenho e §6 Conclusões |

**Papel na solução integrada.** O módulo IoT mede a profundidade de irregularidades com um sensor no veículo; este módulo detecta buracos em fotos e vídeos; cada detecção com confiança ≥ 0,5 é gravada no banco compartilhado (`citizentech.db`, tabela `deteccoes_visao`) e passa a contar na pontuação de prioridade do trecho (`vw_prioridade_trechos`), que o agente GenAI usa para montar o relatório de inspeção.

> A detecção automática apoia a **triagem** e a otimização das rotas de inspeção. Ela não substitui a vistoria técnica presencial para atestar a gravidade real.

In [ ]:
import os, sys, json, time, re, glob, collections, platform, subprocess, random
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, psutil, matplotlib.pyplot as plt
from IPython.display import Markdown, display
import ultralytics
from ultralytics import YOLO

VISAO = Path.cwd().resolve()
if not (VISAO / "detector.py").exists():
    VISAO = (Path.cwd() / "citizentech" / "visao").resolve()
CITIZENTECH, DATASET, EVID = VISAO.parent, VISAO / "dataset", VISAO / "evidencias"
EVID.mkdir(exist_ok=True)
sys.path.insert(0, str(VISAO)); sys.path.insert(0, str(CITIZENTECH))
from detector import DetectorBuracos, medir_latencia, desenhar, Deteccao, CONF_REGISTRO, CONF_EXIBICAO
import banco

# paleta e estilo dos gráficos
PAL = {"azul": "#2a78d6", "laranja": "#eb6834", "aqua": "#1baf7a", "amarelo": "#eda100",
       "vermelho": "#e34948", "cinza": "#898781", "grade": "#e1e0d9", "ink": "#0b0b0b"}
plt.rcParams.update({"figure.dpi": 110, "font.family": "sans-serif", "axes.titleweight": "bold", "axes.titlesize": 11})
def estilo(ax, titulo=None, xlabel=None, ylabel=None):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color("#c3c2b7")
    ax.grid(axis="y", color=PAL["grade"], linewidth=0.8); ax.set_axisbelow(True)
    ax.tick_params(colors=PAL["cinza"], labelsize=9)
    if titulo: ax.set_title(titulo, loc="left", color=PAL["ink"])
    if xlabel: ax.set_xlabel(xlabel, color=PAL["cinza"])
    if ylabel: ax.set_ylabel(ylabel, color=PAL["cinza"])

def cpu_nome():
    try:
        for l in open("/proc/cpuinfo"):
            if "model name" in l: return l.split(":", 1)[1].strip()
    except Exception: pass
    return platform.processor() or "desconhecido"
HARDWARE = {"cpu": cpu_nome(), "nucleos_fisicos": psutil.cpu_count(logical=False), "threads": psutil.cpu_count(),
            "ram_gb": round(psutil.virtual_memory().total / 1e9, 1), "gpu": "nenhuma (CUDA indisponível)" if not torch.cuda.is_available() else torch.cuda.get_device_name(0),
            "so": platform.platform(), "python": platform.python_version(), "torch": torch.__version__,
            "ultralytics": ultralytics.__version__, "opencv": cv2.__version__, "threads_torch": torch.get_num_threads()}
pd.DataFrame([HARDWARE]).T.rename(columns={0: "valor"})

## 1. Dados

### 1.1 Bases públicas escolhidas
| Base | Uso | Licença | Conteúdo |
|---|---|---|---|
| `rupesh002/pothole-detection-dataset` (Hugging Face; exportação Roboflow "pothole.v1i") | treino / validação / teste | MIT | 715 imagens 640×640 de buracos em vias, 1 classe, rótulos YOLO (1.269 caixas) |
| `Ryukijano/Pothole-detection-Yolov8` (Hugging Face; amostra da base Roboflow "Potholes Detection") | **teste externo** (generalização) | CC BY 4.0 | split de teste com 100 imagens 640×640, deduplicado para 88 fotos de origem |

As exportações do Roboflow contêm **variações aumentadas da mesma foto** (espelhamento horizontal), identificadas pelo sufixo `.rf.<hash>` no nome do arquivo. Se essas variações caíssem em conjuntos diferentes, o teste ficaria contaminado. O script `preparar_dados.py` agrupa os arquivos pela **imagem de origem** e divide 70 / 15 / 15 **por grupo**; validação e teste ficam com uma única variante por foto.

In [ ]:
if not (DATASET / "data.yaml").exists():
    subprocess.run([sys.executable, str(VISAO / "preparar_dados.py")], check=True)
print((DATASET / "data.yaml").read_text())

RE_FONTE = re.compile(r"(?:_jpg|_png|_jpeg)?\.rf\.[0-9a-f]+\.(?:jpg|png|jpeg)$", re.I)
fonte = lambda n: RE_FONTE.sub("", n)
def ler_gt(lbl, w=640, h=640):
    out = []
    for l in Path(lbl).read_text().splitlines():
        if not l.strip(): continue
        _, cx, cy, bw, bh = map(float, l.split()[:5])
        out.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
    return out

linhas, fontes = [], {}
for split in ("train", "val", "test", "test_externo"):
    imgs = sorted((DATASET / split / "images").glob("*.jpg"))
    gts = [ler_gt(DATASET / split / "labels" / (p.stem + ".txt")) for p in imgs]
    fontes[split] = {fonte(p.name) for p in imgs}
    tam = collections.Counter(cv2.imread(str(p)).shape[:2] for p in imgs[:20])
    linhas.append({"conjunto": split, "imagens": len(imgs), "fotos de origem": len(fontes[split]), "caixas": sum(map(len, gts)),
                   "caixas/imagem": round(sum(map(len, gts)) / len(imgs), 2), "resolução": ", ".join(f"{h}x{w}" for (h, w) in tam)})
vaz = {f"{a}∩{b}": len(fontes[a] & fontes[b]) for a, b in (("train", "val"), ("train", "test"), ("val", "test"), ("train", "test_externo"))}
print("Fotos de origem compartilhadas entre conjuntos (deve ser 0):", vaz)
tabela_dados = pd.DataFrame(linhas); tabela_dados

### 1.2 Exemplos rotulados
Caixas verdes = rótulo humano (ground truth). As imagens já vêm redimensionadas para 640×640 (stretch) pela exportação; o Ultralytics ainda aplica *letterbox* para 640 e normaliza os pixels para o intervalo 0–1 na entrada da rede.

In [ ]:
random.seed(7)
amostra = random.sample(sorted((DATASET / "train" / "images").glob("*.jpg")), 8)
fig, axs = plt.subplots(2, 4, figsize=(13, 6.6))
for ax, p in zip(axs.flat, amostra):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2 in ler_gt(DATASET / "train" / "labels" / (p.stem + ".txt")):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (27, 175, 122), 3)
    ax.imshow(im); ax.set_title(fonte(p.name), fontsize=8, loc="left"); ax.axis("off")
plt.suptitle("Amostras de treino com rótulos (verde = buraco anotado)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.savefig(EVID / "amostras_treino.png", bbox_inches="tight"); plt.show()

### 1.3 Distribuição das caixas
O tamanho relativo das caixas orienta a escolha da resolução de entrada: muitos buracos ocupam menos de 10 % da largura da imagem, o que favorece manter 640 px em vez de reduzir para 320 px.

In [ ]:
areas, por_img = [], []
for split in ("train", "val", "test"):
    for lbl in (DATASET / split / "labels").glob("*.txt"):
        g = ler_gt(lbl); por_img.append(len(g))
        areas += [np.sqrt((x2 - x1) * (y2 - y1)) / 640 for x1, y1, x2, y2 in g]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.hist(areas, bins=30, color=PAL["azul"], edgecolor="white", linewidth=0.5)
estilo(a1, "Tamanho relativo das caixas (√área / lado da imagem)", "fração do lado da imagem", "caixas")
cont = collections.Counter(min(n, 6) for n in por_img)
a2.bar([str(k) if k < 6 else "6+" for k in sorted(cont)], [cont[k] for k in sorted(cont)], color=PAL["azul"], width=0.6)
estilo(a2, "Caixas por imagem (treino+val+teste)", "nº de buracos na imagem", "imagens")
plt.tight_layout(); plt.savefig(EVID / "distribuicao_caixas.png", bbox_inches="tight"); plt.show()
print(f"mediana do lado relativo: {np.median(areas):.3f} | 25% das caixas abaixo de {np.percentile(areas, 25):.3f} do lado da imagem")

## 2. Modelo e treinamento

### 2.1 Escolha da arquitetura
- **Tarefa:** localização por caixas (não só classificação), porque a prefeitura precisa saber **onde** está o buraco no quadro e quantos existem.
- **Detector:** **YOLO11n** (Ultralytics 8.4) — detector de um estágio, *anchor-free*, ~2,6 M de parâmetros e ~6,5 GFLOPs. É a menor variante da família e foi escolhida porque o protótipo precisa rodar em **CPU comum** (sem GPU) com várias imagens por segundo, e porque um modelo desse porte cabe em um computador de bordo ou numa câmera veicular.
- **Transfer learning:** partimos dos pesos pré-treinados no COCO e ajustamos **todas as camadas** na base de buracos (fine-tuning completo), que é pequena (502 imagens de treino). Treinar do zero não convergiria com esse volume de dados.
- **Pré-processamento e aumento de dados** (Ultralytics, apenas no treino): *letterbox* 640, normalização 0–1, mosaico (4 imagens), variação de matiz/saturação/brilho (HSV), translação e escala aleatórias, espelhamento horizontal. O mosaico é desligado nas últimas 5 épocas (`close_mosaic`).
- **Hiperparâmetros:** 30 épocas, lote 8, otimizador escolhido automaticamente (AdamW, lr 0,002), *early stopping* com paciência 10, semente 42.

O treinamento foi executado pelo script `treinar.py` (comando abaixo). Como leva ~2–3 h em CPU, o notebook reutiliza os pesos salvos em `modelos/best.pt`; mude `TREINAR = True` para repetir.

In [ ]:
TREINAR = False
RUN = VISAO / "runs" / "yolo11n_buracos"
PESOS = VISAO / "modelos" / "best.pt"
print("Comando usado:  python citizentech/visao/treinar.py --epochs 30 --batch 8 --imgsz 640")
if TREINAR or not PESOS.exists():
    subprocess.run([sys.executable, str(VISAO / "treinar.py"), "--epochs", "30", "--batch", "8", "--imgsz", "640"], check=True)
import yaml
args = yaml.safe_load((RUN / "args.yaml").read_text())
chaves = ["model", "data", "epochs", "batch", "imgsz", "optimizer", "lr0", "momentum", "weight_decay", "patience", "close_mosaic",
          "mosaic", "fliplr", "hsv_h", "hsv_s", "hsv_v", "scale", "translate", "seed", "device", "workers"]
pd.DataFrame({"parâmetro": chaves, "valor": [args.get(k) for k in chaves]})

In [ ]:
model = YOLO(str(PESOS))
n_camadas, n_params, _, gflops = model.info(verbose=False)
print(f"YOLO11n ajustado: {n_camadas} camadas · {n_params/1e6:.2f} M parâmetros · {gflops:.1f} GFLOPs (640x640) · classes: {model.names}")
res = pd.read_csv(RUN / "results.csv"); res.columns = [c.strip() for c in res.columns]
melhor = res.loc[res["metrics/mAP50-95(B)"].idxmax()]
print(f"Épocas executadas: {len(res)} · melhor época (mAP50-95 de validação): {int(melhor['epoch'])}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(res["epoch"], res["train/box_loss"], color=PAL["azul"], lw=2, label="box (treino)")
a1.plot(res["epoch"], res["val/box_loss"], color=PAL["azul"], lw=2, ls="--", label="box (val)")
a1.plot(res["epoch"], res["train/cls_loss"], color=PAL["laranja"], lw=2, label="cls (treino)")
a1.plot(res["epoch"], res["val/cls_loss"], color=PAL["laranja"], lw=2, ls="--", label="cls (val)")
estilo(a1, "Perdas por época", "época", "perda"); a1.legend(frameon=False, fontsize=8)
a2.plot(res["epoch"], res["metrics/mAP50(B)"], color=PAL["azul"], lw=2, label="mAP@0,5")
a2.plot(res["epoch"], res["metrics/mAP50-95(B)"], color=PAL["laranja"], lw=2, label="mAP@0,5:0,95")
a2.plot(res["epoch"], res["metrics/precision(B)"], color=PAL["aqua"], lw=1.5, label="precisão")
a2.plot(res["epoch"], res["metrics/recall(B)"], color=PAL["amarelo"], lw=1.5, label="recall")
a2.set_ylim(0, 1); estilo(a2, "Métricas de validação por época", "época"); a2.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(EVID / "curvas_treino.png", bbox_inches="tight"); plt.show()

## 3. Avaliação

### 3.1 Métricas no conjunto de teste (93 fotos nunca vistas) e no teste externo (88 fotos de outra base)
Métricas padrão de detecção calculadas pelo Ultralytics: **precisão**, **recall**, **mAP@0,5** e **mAP@0,5:0,95** (IoU de 0,5 a 0,95). A matriz de confusão usa confiança ≥ 0,25 e IoU ≥ 0,45; para uma classe ela se reduz a verdadeiros positivos, falsos positivos (caixa sem buraco anotado) e falsos negativos (buraco anotado sem caixa).

In [ ]:
def avaliar(data_yaml, split, nome):
    m = model.val(data=str(data_yaml), split=split, imgsz=640, batch=8, device="cpu", plots=True,
                  project=str(VISAO / "runs"), name=nome, exist_ok=True, verbose=False)
    cm = m.confusion_matrix.matrix
    return {"conjunto": nome, "precisão": m.box.mp, "recall": m.box.mr, "mAP@0,5": m.box.map50, "mAP@0,5:0,95": m.box.map,
            "VP": int(cm[0, 0]), "FP": int(cm[0, 1]), "FN": int(cm[1, 0])}
r_teste = avaliar(DATASET / "data.yaml", "test", "val_teste")
r_ext = avaliar(DATASET / "data_externo.yaml", "val", "val_teste_externo")
metricas = pd.DataFrame([r_teste, r_ext]).set_index("conjunto")
metricas.style.format({"precisão": "{:.3f}", "recall": "{:.3f}", "mAP@0,5": "{:.3f}", "mAP@0,5:0,95": "{:.3f}"})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (nome, r) in zip(axs, (("teste (mesma base)", r_teste), ("teste externo (outra base)", r_ext))):
    m = np.array([[r["VP"], r["FP"]], [r["FN"], 0]])
    ax.imshow(m, cmap="Blues", vmin=0, vmax=max(1, m.max()))
    for (i, j), v in np.ndenumerate(m):
        if (i, j) != (1, 1): ax.text(j, i, str(v), ha="center", va="center", fontsize=14, color="white" if v > m.max() * 0.6 else PAL["ink"])
    ax.set_xticks([0, 1]); ax.set_xticklabels(["buraco (real)", "fundo (real)"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["buraco (previsto)", "fundo (previsto)"])
    ax.set_title(f"Matriz de confusão — {nome}", loc="left", fontweight="bold"); ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.savefig(EVID / "matriz_confusao.png", bbox_inches="tight"); plt.show()
from IPython.display import Image as IPImage
display(Markdown("**Curva precisão × recall (teste, gerada pelo Ultralytics):**"))
display(IPImage(filename=str(VISAO / "runs" / "val_teste" / "BoxPR_curve.png"), width=520))

### 3.2 Escolha do limiar de confiança
Para cada limiar de confiança calculamos precisão, recall e F1 no conjunto de teste, casando previsões com rótulos por IoU ≥ 0,5 (pareamento guloso por confiança). O limiar define o equilíbrio entre **alarmes falsos** (custo: deslocamento inútil da equipe) e **buracos perdidos** (custo: risco ao cidadão).

Adotamos dois limiares na integração: **0,25 para exibir** a caixa (amarela, "incerta") e **0,50 para registrar** no banco e pontuar o trecho (vermelha, "confirmada"). Como o módulo faz triagem e a vistoria presencial confirma, preferimos recall alto na exibição e precisão maior no que entra na pontuação.

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def prever_tudo(split):
    imgs = sorted((DATASET / split / "images").glob("*.jpg"))
    preds, gts = {}, {}
    for r in model.predict([str(p) for p in imgs], conf=0.001, imgsz=640, device="cpu", stream=True, verbose=False):
        k = Path(r.path).name
        b = r.boxes.xyxy.cpu().numpy(); c = r.boxes.conf.cpu().numpy()
        preds[k] = sorted([[*bb, cc] for bb, cc in zip(b.tolist(), c.tolist())], key=lambda x: -x[4])
        gts[k] = ler_gt(DATASET / split / "labels" / (Path(k).stem + ".txt"))
    return preds, gts

def casar(preds, gts, thr, iou_thr=0.5):
    tp = fp = fn = 0; detalhes = {}
    for k, g in gts.items():
        usados, fps, tps = set(), [], []
        for box in preds.get(k, []):
            if box[4] < thr: continue
            best, bi = 0, -1
            for j, gb in enumerate(g):
                if j in usados: continue
                v = iou(box[:4], gb)
                if v > best: best, bi = v, j
            if best >= iou_thr: tp += 1; usados.add(bi); tps.append(box)
            else: fp += 1; fps.append(box)
        fns = [gb for j, gb in enumerate(g) if j not in usados]; fn += len(fns)
        detalhes[k] = {"tp": tps, "fp": fps, "fn": fns}
    P = tp / (tp + fp) if tp + fp else 0.0; R = tp / (tp + fn) if tp + fn else 0.0
    return {"limiar": thr, "precisão": P, "recall": R, "F1": 2 * P * R / (P + R) if P + R else 0.0, "VP": tp, "FP": fp, "FN": fn}, detalhes

preds_teste, gts_teste = prever_tudo("test")
limiares = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = pd.DataFrame([casar(preds_teste, gts_teste, t)[0] for t in limiares])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(sweep["limiar"], sweep["precisão"], color=PAL["azul"], lw=2, label="precisão")
ax.plot(sweep["limiar"], sweep["recall"], color=PAL["laranja"], lw=2, label="recall")
ax.plot(sweep["limiar"], sweep["F1"], color=PAL["aqua"], lw=2, label="F1")
for t, lab in ((CONF_EXIBICAO, "exibição"), (CONF_REGISTRO, "registro")):
    ax.axvline(t, color=PAL["cinza"], lw=1, ls=":"); ax.text(t + 0.01, 0.03, f"{lab} ({t})", color=PAL["cinza"], fontsize=8)
ax.set_ylim(0, 1); estilo(ax, "Precisão, recall e F1 por limiar de confiança (teste, IoU ≥ 0,5)", "limiar de confiança"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(EVID / "limiar_confianca.png", bbox_inches="tight"); plt.show()
melhor_f1 = sweep.loc[sweep["F1"].idxmax()]
print(f"Melhor F1 = {melhor_f1['F1']:.3f} no limiar {melhor_f1['limiar']}")
sweep[sweep["limiar"].isin([0.1, 0.25, 0.4, 0.5, 0.6, 0.75])].round(3)

### 3.3 Análise de acertos e erros
Com o limiar de registro (0,5): **verde** = rótulo humano, **vermelho** = detecção correta (IoU ≥ 0,5), **laranja** = falso positivo, **"FN"** = buraco anotado que o modelo não encontrou. As imagens abaixo são as que concentram mais erros no conjunto de teste, seguidas de exemplos de acerto limpo.

In [ ]:
_, det05 = casar(preds_teste, gts_teste, CONF_REGISTRO)
ordem = sorted(det05, key=lambda k: -(len(det05[k]["fp"]) + len(det05[k]["fn"])))
piores = ordem[:6]
limpos = [k for k in ordem[::-1] if det05[k]["tp"] and not det05[k]["fp"] and not det05[k]["fn"]][:3]

def pintar(k, info):
    im = cv2.cvtColor(cv2.imread(str(DATASET / "test" / "images" / k)), cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2 in gts_teste[k]: cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (27, 175, 122), 2)
    for x1, y1, x2, y2, c in info["tp"]: cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (227, 73, 72), 3); cv2.putText(im, f"{c:.2f}", (int(x1)+3, int(y1)+18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (227, 73, 72), 2)
    for x1, y1, x2, y2, c in info["fp"]: cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (235, 104, 52), 3); cv2.putText(im, f"FP {c:.2f}", (int(x1)+3, int(y1)+18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (235, 104, 52), 2)
    for x1, y1, x2, y2 in info["fn"]: cv2.putText(im, "FN", (int(x1)+3, int(y2)-6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (27, 175, 122), 2)
    return im

fig, axs = plt.subplots(3, 3, figsize=(13, 13))
for ax, k in zip(axs.flat, piores + limpos):
    i = det05[k]; ax.imshow(pintar(k, i)); ax.axis("off")
    ax.set_title(f"{fonte(k)} — VP {len(i['tp'])} · FP {len(i['fp'])} · FN {len(i['fn'])}", fontsize=9, loc="left")
plt.suptitle("Linhas 1–2: imagens com mais erros · linha 3: acertos limpos (limiar 0,5)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.savefig(EVID / "analise_erros.png", bbox_inches="tight"); plt.show()
tot = casar(preds_teste, gts_teste, CONF_REGISTRO)[0]
print(f"No limiar 0,5: {tot['VP']} VP, {tot['FP']} FP, {tot['FN']} FN em {len(gts_teste)} imagens · "
      f"imagens com algum erro: {sum(1 for k in det05 if det05[k]['fp'] or det05[k]['fn'])}")

**Padrões observados nos erros** (a comentar junto com as imagens acima):
- **Falsos negativos** concentram-se em buracos **pequenos ou distantes**, em buracos **parcialmente cobertos por água ou sombra**, e em grupos de buracos muito próximos, em que o modelo funde dois rótulos em uma caixa só (o rótulo humano conta dois, o modelo acerta um e perde o outro).
- **Falsos positivos** aparecem em **remendos escuros**, **manchas de óleo**, **tampas de bueiro** e **sombras com borda bem definida**, que compartilham a textura "mancha escura sobre asfalto". Em alguns casos o "FP" é um buraco real que o anotador não marcou, o que é uma limitação do próprio rótulo.
- **Variação de iluminação** forte (contraluz, asfalto molhado refletindo o céu) reduz a confiança: várias detecções corretas ficam entre 0,25 e 0,5, por isso elas são exibidas como "incertas" na integração e não entram na pontuação.

## 4. Desempenho computacional
Latência média por imagem (inferência pura, lote 1, incluindo pré e pós-processamento do Ultralytics) e taxa de quadros resultante, medidas na CPU descrita no início, para três resoluções de entrada. Não há GPU na máquina de teste.

In [ ]:
det = DetectorBuracos(PESOS, imgsz=640)
imgs_bench = [str(p) for p in sorted((DATASET / "test" / "images").glob("*.jpg"))[:30]]
bench = pd.DataFrame([medir_latencia(det, imgs_bench, imgsz=s) for s in (320, 480, 640)])
bench["hardware"] = HARDWARE["cpu"]
display(bench.round(1))
fig, ax = plt.subplots(figsize=(7, 3.4))
barras = ax.bar(bench["imgsz"].astype(str), bench["latencia_media_ms"], color=PAL["azul"], width=0.5)
for b, (lat, fps) in zip(barras, zip(bench["latencia_media_ms"], bench["fps"])):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, f"{lat:.0f} ms · {fps:.1f} FPS", ha="center", fontsize=9, color=PAL["ink"])
estilo(ax, f"Latência por imagem em CPU ({HARDWARE['nucleos_fisicos']} núcleos, sem GPU)", "resolução de entrada (px)", "ms por imagem")
ax.set_ylim(0, bench["latencia_media_ms"].max() * 1.25)
plt.tight_layout(); plt.savefig(EVID / "latencia.png", bbox_inches="tight"); plt.show()

**Equilíbrio qualidade × velocidade.** Mantivemos 640 px na integração porque a base tem muitos buracos pequenos (§1.3) e a diferença de latência ainda permite processar vídeo em tempo quase real com `stride` (1 a cada N quadros) numa CPU modesta. Em um dispositivo de borda mais fraco, 480 px seria a primeira concessão; 320 px só para pré-triagem. Exportar para ONNX/OpenVINO e quantizar em INT8 são otimizações futuras para a câmera veicular.

## 5. Integração ao CitizenTech

### 5.1 Imagem estática → detecções com confiança → banco compartilhado
`DetectorBuracos.detectar_imagem()` devolve as caixas e a confiança e desenha a anotação; `registrar_no_banco()` grava cada detecção confirmada em `deteccoes_visao`. A pontuação do trecho muda imediatamente na view usada pelo agente GenAI.

In [ ]:
conn = banco.conectar(); banco.inicializar(conn)
conn.execute("DELETE FROM deteccoes_visao WHERE arquivo LIKE 'nb_%'"); conn.commit()   # idempotência do notebook
TRECHO_FOTO = "SIM0004"   # Av. Jabaquara: trecho ainda sem nenhum registro
antes = banco.prioridade_do_trecho(conn, TRECHO_FOTO)
exemplo = next(k for k in ordem[::-1] if len(det05[k]["tp"]) >= 2)      # uma foto com 2+ acertos
r = det.detectar_imagem(str(DATASET / "test" / "images" / exemplo))
plt.figure(figsize=(7, 7)); plt.imshow(cv2.cvtColor(r.imagem_anotada, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"Entrada: {fonte(exemplo)} · {len(r.confirmadas)} confirmadas, {len(r.deteccoes)-len(r.confirmadas)} incertas · {r.latencia_ms:.0f} ms", loc="left", fontsize=10, fontweight="bold")
plt.savefig(EVID / "integracao_imagem.png", bbox_inches="tight"); plt.show()
display(pd.DataFrame([{"classe": d.classe, "confiança": round(d.confianca, 3), "bbox": d.bbox, "registra?": d.confianca >= CONF_REGISTRO} for d in r.deteccoes]))
n = det.registrar_no_banco(conn, TRECHO_FOTO, "nb_" + exemplo, r.deteccoes)
depois = banco.prioridade_do_trecho(conn, TRECHO_FOTO)
print(f"{n} detecções gravadas em deteccoes_visao para {depois['nome']}")
pd.DataFrame([{"momento": "antes", **{k: antes[k] for k in ("deteccoes_visao", "pontuacao", "nivel")}},
              {"momento": "depois", **{k: depois[k] for k in ("deteccoes_visao", "pontuacao", "nivel")}}])

### 5.2 Vídeo / sequência de quadros → monitoramento contínuo
`DetectorBuracos.processar_video()` roda o detector com **rastreamento ByteTrack**: cada buraco recebe um identificador e é gravado **uma única vez** no banco (com a maior confiança observada), para que a pontuação do trecho não seja inflada por uma detecção por quadro. O vídeo anotado mostra as caixas, a confiança, o contador de buracos e o FPS medido.

Se houver um vídeo real em `midia/` (gravado na rua), ele é usado; caso contrário o notebook monta uma sequência de quadros a partir de fotos do conjunto de teste, com movimento de câmera simulado, para demonstrar o fluxo.

In [ ]:
MIDIA = CITIZENTECH.parent / "midia"
videos = sorted([p for ext in ("*.mp4", "*.MP4", "*.mov", "*.MOV", "*.avi") for p in MIDIA.glob(ext)])
if videos:
    VIDEO, TRECHO_VIDEO, STRIDE = videos[0], "SIM0005", 2
    if not banco.trecho_existe(conn, TRECHO_VIDEO):
        banco.cadastrar_trecho(conn, TRECHO_VIDEO, "Via filmada (vídeo real)", "—")
    print("Vídeo real encontrado:", VIDEO.name)
else:
    VIDEO, TRECHO_VIDEO, STRIDE = VISAO / "saida" / "nb_sequencia_teste.mp4", "SIM0001", 1
    VIDEO.parent.mkdir(exist_ok=True)
    random.seed(3); fotos = random.sample(sorted((DATASET / "test_externo" / "images").glob("*.jpg")), 24)
    vw = cv2.VideoWriter(str(VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), 10, (960, 540))
    for p in fotos:                       # "câmera" avançando: recorte 16:9 com zoom progressivo
        im = cv2.imread(str(p)); h, w = im.shape[:2]
        for i in range(12):
            z = 1.0 + 0.35 * i / 11; cw, ch = int(w / z), int(w / z * 9 / 16)
            x0, y0 = (w - cw) // 2, min(h - ch, int((h - ch) * 0.55 + 0.3 * ch * i / 11))
            vw.write(cv2.resize(im[y0:y0 + ch, x0:x0 + cw], (960, 540)))
    vw.release(); print("Sequência sintética criada a partir de 24 fotos do teste externo:", VIDEO.name)

conn.execute("DELETE FROM deteccoes_visao WHERE arquivo = ? AND trecho_uid = ?", (VIDEO.name, TRECHO_VIDEO)); conn.commit()
antes_v = banco.prioridade_do_trecho(conn, TRECHO_VIDEO)
SAIDA = VISAO / "saida" / (VIDEO.stem + "_anotado.mp4")
rv = det.processar_video(VIDEO, SAIDA, stride=STRIDE, trecho=TRECHO_VIDEO, conn=conn, progresso=False)
depois_v = banco.prioridade_do_trecho(conn, TRECHO_VIDEO)
resumo_video = {"vídeo": VIDEO.name, "resolução": f"{rv.largura}x{rv.altura}", "fps do vídeo": round(rv.fps_video, 1), "quadros lidos": rv.frames_lidos,
                "quadros processados": rv.frames_processados, "stride": STRIDE, "latência média (ms/quadro)": round(rv.latencia_media_ms, 1),
                "FPS de processamento": round(rv.fps_processamento, 2), "tempo total (s)": round(rv.tempo_total_s, 1),
                "buracos rastreados": len(rv.buracos_rastreados), "buracos confirmados (≥0,5)": len(rv.buracos_confirmados),
                "registros gravados no banco": rv.registros_banco, "trecho": f"{TRECHO_VIDEO} · {depois_v['nome']}",
                "pontuação antes → depois": f"{antes_v['pontuacao']} ({antes_v['nivel']}) → {depois_v['pontuacao']} ({depois_v['nivel']})"}
display(pd.DataFrame([resumo_video]).T.rename(columns={0: "valor"}))
print("Vídeo anotado salvo em", SAIDA)

In [ ]:
cap = cv2.VideoCapture(str(SAIDA)); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axs = plt.subplots(2, 2, figsize=(13, 7.6))
for ax, f in zip(axs.flat, np.linspace(total * 0.1, total * 0.9, 4).astype(int)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, fr = cap.read()
    if ok: ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    ax.set_title(f"quadro {f}", fontsize=9, loc="left"); ax.axis("off")
cap.release()
plt.suptitle("Quadros do vídeo anotado (monitoramento contínuo com rastreamento)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.savefig(EVID / "integracao_video.png", bbox_inches="tight"); plt.show()

### 5.3 Efeito no ranking de prioridade (o que o agente GenAI e a zeladoria veem)

In [ ]:
rank = pd.DataFrame(banco.prioridades(conn, apenas_com_registro=False))[["uid", "nome", "graves", "leves", "reportes", "deteccoes_visao", "conf_media_visao", "umidade_media", "pontuacao", "nivel"]]
display(rank)
fig, ax = plt.subplots(figsize=(8, 3.2))
cores = [PAL["vermelho"] if n == "alta" else PAL["amarelo"] if n == "media" else PAL["azul"] for n in rank["nivel"]]
ax.barh(rank["nome"][::-1], rank["pontuacao"][::-1], color=cores[::-1], height=0.55)
for y, (p, n) in enumerate(zip(rank["pontuacao"][::-1], rank["nivel"][::-1])): ax.text(p + 0.1, y, f"{p} · {n}", va="center", fontsize=9)
ax.axvline(4, color=PAL["cinza"], lw=1, ls=":"); ax.axvline(8, color=PAL["cinza"], lw=1, ls=":")
estilo(ax, "Pontuação de prioridade por trecho após a integração IoT + Visão", "pontuação"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.savefig(EVID / "ranking_apos_visao.png", bbox_inches="tight"); plt.show()

## 6. Conclusões

In [ ]:
resultado = {
    "modelo": "YOLO11n (Ultralytics 8.4) ajustado a partir do COCO", "parametros_M": round(n_params / 1e6, 2), "gflops": round(gflops, 1),
    "epocas": int(len(res)), "melhor_epoca": int(melhor["epoch"]), "imgsz": 640, "dataset": tabela_dados.to_dict("records"),
    "teste": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r_teste.items()},
    "teste_externo": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r_ext.items()},
    "limiar": {"exibicao": CONF_EXIBICAO, "registro": CONF_REGISTRO, "melhor_f1": {"limiar": float(melhor_f1["limiar"]), "F1": round(float(melhor_f1["F1"]), 4)},
               "em_0_5": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in tot.items()}},
    "latencia": bench.round(2).to_dict("records"), "hardware": HARDWARE, "video": resumo_video,
}
(EVID / "metricas_visao.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=1, default=str))
l640 = bench[bench["imgsz"] == 640].iloc[0]
display(Markdown(f'''
| Resultado | Valor |
|---|---|
| Teste (mesma base, 93 fotos) | precisão **{r_teste["precisão"]:.3f}** · recall **{r_teste["recall"]:.3f}** · mAP@0,5 **{r_teste["mAP@0,5"]:.3f}** · mAP@0,5:0,95 **{r_teste["mAP@0,5:0,95"]:.3f}** |
| Teste externo (outra base, 88 fotos) | precisão **{r_ext["precisão"]:.3f}** · recall **{r_ext["recall"]:.3f}** · mAP@0,5 **{r_ext["mAP@0,5"]:.3f}** · mAP@0,5:0,95 **{r_ext["mAP@0,5:0,95"]:.3f}** |
| Limiar de registro 0,5 (teste, IoU ≥ 0,5) | precisão {tot["precisão"]:.3f} · recall {tot["recall"]:.3f} · F1 {tot["F1"]:.3f} (melhor F1 {melhor_f1["F1"]:.3f} em {melhor_f1["limiar"]}) |
| Latência em CPU ({HARDWARE["cpu"]}) | **{l640["latencia_media_ms"]:.0f} ms/imagem a 640 px ({l640["fps"]:.1f} FPS)** · {bench[bench["imgsz"]==480].iloc[0]["latencia_media_ms"]:.0f} ms a 480 px · {bench[bench["imgsz"]==320].iloc[0]["latencia_media_ms"]:.0f} ms a 320 px |
| Vídeo | {resumo_video["quadros processados"]} quadros processados, {resumo_video["FPS de processamento"]} FPS, {resumo_video["buracos confirmados (≥0,5)"]} buracos únicos confirmados e gravados no banco |

Resumo salvo em `evidencias/metricas_visao.json`.
'''))

**O que funcionou.** Com transfer learning a partir do COCO, um detector de 2,6 M de parâmetros atinge mAP@0,5 alto numa base pequena, generaliza para uma segunda base pública e roda em CPU comum com várias imagens por segundo. A integração grava cada detecção confirmada no banco compartilhado e muda o ranking que a zeladoria e o agente GenAI usam; no vídeo, o rastreamento evita contar o mesmo buraco várias vezes.

**Limitações.** (1) A base de treino é pequena e tem pouca variedade de clima e iluminação noturna; (2) sombras, remendos e tampas de bueiro geram falsos positivos; (3) o modelo não mede profundidade nem gravidade: isso vem do sensor IoT e da vistoria; (4) o rastreamento depende de câmera estável; curvas bruscas quebram o identificador e podem duplicar um buraco; (5) as métricas de teste externo mostram que mudar de cidade/câmera reduz o desempenho, o que pede re-treino com fotos locais (por exemplo, as gravadas pela própria frota).

**Próximos passos.** Anotar e incluir fotos das vias de São Paulo gravadas pela frota; treinar o YOLO11s para comparar qualidade × latência; exportar para ONNX/OpenVINO com quantização INT8 para a câmera veicular; combinar a detecção visual com a profundidade medida pelo IoT no mesmo instante (fusão de sensores) para estimar gravidade.